# Groupby (average)

In [ ]:
# Necessary Libraries
import pandas as pd
from datetime import datetime, time, timedelta
import math

## Data Awal

In [ ]:
# Dataframe Creation - Raw Data from PM17
raw17 = pd.read_csv('17-130326.csv', delimiter = ';', encoding = 'utf-16').dropna(inplace = False).reset_index(drop = True)
raw17.rename(columns={'Speed Yankee': 'Yankee Speed', 'Preasure Yankee': 'Yankee Pressure'}, inplace=True)
raw17.head()

,Time,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner
0,11/03/26 14:02:21,1159.821533,951.01,7.95,1307.45,3.39,30.40,32.25,0.94,215.70
1,11/03/26 14:03:21,1159.821533,951.31,7.94,1309.80,3.39,30.41,32.26,0.94,215.39
2,11/03/26 14:04:21,1159.909790,951.38,7.97,1312.64,3.40,30.40,32.25,0.94,216.15
3,11/03/26 14:05:21,1159.909790,951.16,8.04,1309.84,3.40,30.40,32.26,0.94,215.83
4,11/03/26 14:06:21,1159.997803,951.31,8.06,1309.14,3.40,30.40,32.26,0.94,215.95


In [ ]:
raw17.info()

<class 'pandas.DataFrame'>
RangeIndex: 2570 entries, 0 to 2569
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Time               2570 non-null   str    
 1   Yankee Speed       2570 non-null   float64
 2   Pope Reel Speed    2570 non-null   float64
 3   Yankee Pressure    2570 non-null   float64
 4   Stock Flow         2570 non-null   float64
 5   Stock Consistency  2570 non-null   float64
 6   Flow Coating       2570 non-null   float64
 7   Flow Release       2570 non-null   float64
 8   Jet Wire Ratio     2570 non-null   float64
 9   Load KWH Refiner   2570 non-null   float64
dtypes: float64(9), str(1)
memory usage: 200.9 KB


In [ ]:
# Dataframe Creation - Raw Data Reel
reel_pm17 = pd.read_excel("Data Reel/DATA REEL JAN'26 SD APR'26 (PM12, PM15, PM17).xlsx", sheet_name = 'PM17')
reel_pm17.head()

,Time,Tanggal,Grade,Shift,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,7.15,01.01.26,FC 12.2.Vp,1,30,12.94,0.97,492,193,147,0.298780,30,86.5,Acc Sotiss
1,8.05,01.01.26,FC 12.2.Vp,1,31,12.78,1.03,565,174,158,0.279646,32,86.9,Acc Sotiss
2,8.55,01.01.26,FC 12.2.Vp,1,32,12.92,0.97,538,203,128,0.237918,30,87,Acc Sotiss
3,9.35,01.01.26,FC 12.2.Vp,1,33,12.64,0.95,470,161,137,0.291489,30,87,Acc Sotiss
4,10.15,01.01.26,FC 12.2.Vp,1,34,12.28,0.96,378,131,105,0.277778,30,87.1,Acc Sotiss


### Shift

In [ ]:
shift1_start = time(7, 0, 1)
shift1_end   = time(15, 0, 0)
shift2_start = time(15, 0, 1)
shift2_end   = time(23, 0, 0)
def assign_shift(t):
    if shift1_start <= t <= shift1_end:
        return 'Shift 1'
    elif shift2_start <= t <= shift2_end:
        return 'Shift 2'
    else:
        return 'Shift 3'

### Convert Time Input Reel Time

Function to convert time input reel time to hh:mm format.

In [ ]:
def time_to_hms(val):
    s = str(val).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return pd.NA

    # If already contains colon, parse parts directly
    if ':' in s:
        parts = s.split(':')
        h = int(parts[0])
        m = int(parts[1]) if len(parts) > 1 and parts[1] != '' else 0
        sec = int(parts[2]) if len(parts) > 2 and parts[2] != '' else 0

    # If contains dot, treat left as hours and right as minutes (common human shorthand)
    elif '.' in s:
        left, right = s.split('.', 1)
        if int(left) >= 24:
            return pd.NA  # Invalid hour value
        left = 0 if left == "24" else left  # Handle "24" as "00"
        h = int(left) if left != '' else 0

        # If right part is short (1 or 2 digits) treat it as minutes (e.g., "4.1" -> 4:01, "11.55" -> 11:55)
        if len(right) <= 2:
            m = int(right)
            sec = 0
        else:
            # If right part is longer, treat the whole value as a decimal hour (fallback)
            # e.g., "4.125" -> 4.125 hours -> convert fractional hour to minutes
            f = float(s)
            total_minutes = int(round((f - math.floor(f)) * 60))
            m = total_minutes
            sec = 0

    # No separator: treat as hours only (e.g., "6" -> 06:00:00)
    else:
        h = int(float(s))
        m = 0
        sec = 0

    # Normalize minutes >= 60 into hours
    if m >= 60:
        extra_h = m // 60
        h = (h + extra_h) % 24
        m = m % 60

    return f"{h:02d}:{m:02d}:{sec:02d}"

## Data Preparation

### Preparation Process

#### Data Parameter PM 17

In [ ]:
urutan_params_pm = [
    'Date', 'Time', 'Shift', 'Join_Key', 'Timestamp',
    'Creping', 'Yankee Speed', 'Pope Reel Speed',
    'Yankee Pressure', 'Stock Flow', 'Stock Consistency',
    'Flow Coating', 'Flow Release', 'Jet Wire Ratio',
    'Load KWH Refiner','Key_Date'
]

In [ ]:
def preprocess_pm17(df_input):
    df = df_input.copy()
    df = df.drop(columns=['Time'])
    df.columns = df.columns.str.strip()
    df.insert(0, 'Creping', (df['Yankee Speed'] - df['Pope Reel Speed']) * 100 / df['Yankee Speed'])
    df.insert(0, 'Time', df_input['Time'].apply(lambda x: datetime.strptime(x.split(' ')[1], '%H:%M:%S').time()))
    df['Shift'] = df['Time'].apply(assign_shift)
    
    df.insert(0, 'Date', df_input['Time'].apply(lambda x: x.split(' ')[0]))
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%y')
    df['Key_Date'] = [df['Date'][index] - timedelta(days=1) 
                if
                df['Shift'][index] == 'Shift 3' 
                else df['Date'][index] 
                for index in range(len(df['Shift']))]
    df['Join_Key'] = df['Key_Date'].astype(str) + ' ' + df['Time'].astype(str) + ' ' + df['Shift']
    df['Timestamp'] = df['Date'].astype(str) + ' ' + df['Time'].astype(str)
    df['Shift'] = df['Shift'].str.extract(r'(\d+)').astype(int)
    df = df[urutan_params_pm]
    df.drop(columns=['Key_Date'], inplace=True)
    return df

In [ ]:
pm_17 = preprocess_pm17(raw17)

In [ ]:
num = 700
pm_17.iloc[num:num+2,:]

,Date,Time,Shift,Join_Key,Timestamp,Creping,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner
700,2026-03-12,01:42:21,3,2026-03-11 01:42:21 Shift 3,2026-03-12 01:42:21,17.293809,1159.997803,959.39,7.96,1309.29,3.41,30.40,34.27,0.94,219.45
701,2026-03-12,01:43:21,3,2026-03-11 01:43:21 Shift 3,2026-03-12 01:43:21,17.306740,1159.997803,959.24,7.90,1309.47,3.42,30.41,34.27,0.94,218.78


#### Data Reel

In [ ]:
def preprocess_reel(df_input):
    df = df_input.copy()
    df['Time'] = df['Time'].apply(time_to_hms)
    df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%d.%m.%y')
    df = df.dropna(subset = ['Time']).reset_index(drop = True)
    df['Timestamp'] = df['Tanggal'].astype(str) + ' ' + df['Time'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S')
    # Edit Kolom
    df = df.rename(columns={'Tanggal': 'Date'})
    first_cols = ['Date', 'Time', 'Shift', 'Timestamp']
    other_cols = [col for col in df.columns if col not in first_cols]
    df = df[first_cols + other_cols]
    return df

In [ ]:
reel_pm17 = preprocess_reel(reel_pm17)

In [ ]:
reel_pm17.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-01-01,07:15:00,1,2026-01-01 07:15:00,FC 12.2.Vp,30,12.94,0.97,492,193,147,0.298780,30,86.5,Acc Sotiss
1,2026-01-01,08:05:00,1,2026-01-01 08:05:00,FC 12.2.Vp,31,12.78,1.03,565,174,158,0.279646,32,86.9,Acc Sotiss
2,2026-01-01,08:55:00,1,2026-01-01 08:55:00,FC 12.2.Vp,32,12.92,0.97,538,203,128,0.237918,30,87,Acc Sotiss
3,2026-01-01,09:35:00,1,2026-01-01 09:35:00,FC 12.2.Vp,33,12.64,0.95,470,161,137,0.291489,30,87,Acc Sotiss
4,2026-01-01,10:15:00,1,2026-01-01 10:15:00,FC 12.2.Vp,34,12.28,0.96,378,131,105,0.277778,30,87.1,Acc Sotiss


### Data Fix (Clear)

In [ ]:
reel_fix = pd.read_excel("reel1.xlsx")
params_fix = pd.read_excel("params1.xlsx")

In [ ]:
reel_fix.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-01-01,07:25:00,1,2026-01-01 07:25:00,N 19.2.LB,74,20.10,0.92,1629,920,461,0.282996,12,36.1,Acc Sotiss
1,2026-01-01,08:15:00,1,2026-01-01 08:15:00,N 19.2.LB,75,19.82,0.87,1607,1111,453,0.281892,11,36.2,Acc Sotiss
2,2026-01-01,09:00:00,1,2026-01-01 09:00:00,N 19.2.LB,76,19.78,0.88,1652,1164,496,0.300242,10,35.3,Acc Sotiss
3,2026-01-01,09:55:00,1,2026-01-01 09:55:00,N 19.2.LB,77,20.18,0.74,1955,1283,555,0.283887,13,36.4,Acc Sotiss
4,2026-01-01,10:45:00,1,2026-01-01 10:45:00,N 19.2.LB,78,19.58,0.96,1492,938,392,0.262735,15,37.5,Acc Sotiss


In [ ]:
params_fix.head()

,Date,Time,Shift,Timestamp,Creping,Yankee Speed,Pope Reel Speed,Yankee Pressure,Stock Flow,Stock Consistency,Flow Coating,Flow Release,Jet Wire Ratio,Load KWH Refiner
0,2026-03-11,14:02:21,1,2026-03-11 14:02:21,18.003764,1159.821533,951.01,7.95,1307.45,3.39,30.40,32.25,0.94,215.70
1,2026-03-11,14:03:21,1,2026-03-11 14:03:21,17.977898,1159.821533,951.31,7.94,1309.80,3.39,30.41,32.26,0.94,215.39
2,2026-03-11,14:04:21,1,2026-03-11 14:04:21,17.978104,1159.909790,951.38,7.97,1312.64,3.40,30.40,32.25,0.94,216.15
3,2026-03-11,14:05:21,1,2026-03-11 14:05:21,17.997071,1159.909790,951.16,8.04,1309.84,3.40,30.40,32.26,0.94,215.83
4,2026-03-11,14:06:21,1,2026-03-11 14:06:21,17.990362,1159.997803,951.31,8.06,1309.14,3.40,30.40,32.26,0.94,215.95


## Pipeline

In [ ]:
# 1. MEMBACA DATA
df_reel = reel_pm17.copy()
df_params = pm_17.copy()

# 2. KONVERSI TIMESTAMP
df_reel['Timestamp'] = pd.to_datetime(df_reel['Timestamp'])
df_params['Timestamp'] = pd.to_datetime(df_params['Timestamp'])

# 3. SORT KEY
# REEL: Jam 00-06 ditambah 1 hari (karena di Excel tanggalnya mundur 1 hari dari params)
def create_sort_key_reel(ts):
    if ts.hour < 7:
        return ts + timedelta(days=1)
    return ts

df_reel['Sort_Key'] = df_reel['Timestamp'].apply(create_sort_key_reel)
df_params['Sort_Key'] = df_params['Timestamp']  # Params tidak perlu adjustment

# 4. FILTER BERDASARKAN SORT_KEY RANGE PARAMS
params_sort_min = df_params['Sort_Key'].min()
params_sort_max = df_params['Sort_Key'].max()

df_reel_filtered = df_reel[
    (df_reel['Sort_Key'] >= params_sort_min) & 
    (df_reel['Sort_Key'] <= params_sort_max)
].copy()

df_reel_filtered = df_reel_filtered.sort_values('Sort_Key').reset_index(drop=True)

# 5. DAFTAR VARIABEL
cols_to_avg = [
    'Creping', 'Yankee Speed', 'Pope Reel Speed', 'Yankee Pressure',
    'Stock Flow', 'Stock Consistency', 'Flow Coating', 'Flow Release',
    'Jet Wire Ratio', 'Load KWH Refiner'
]

# 6. LOOPING GROUPBY AVERAGE
results = []

for i in range(len(df_reel_filtered) - 1):
    start_time = df_reel_filtered['Timestamp'].iloc[i]
    end_time = df_reel_filtered['Timestamp'].iloc[i + 1]
    start_sort = df_reel_filtered['Sort_Key'].iloc[i]
    end_sort = df_reel_filtered['Sort_Key'].iloc[i + 1]
    
    mask = (df_params['Sort_Key'] >= start_sort) & (df_params['Sort_Key'] < end_sort)
    df_filtered = df_params.loc[mask]
    
    if len(df_filtered) == 0:
        continue
    
    row = {
        'Start_Time': start_time,
        'End_Time': end_time,
        'Data_Count': len(df_filtered)
    }
    
    for col in cols_to_avg:
        mean_val = df_filtered[col].mean()
        row[f'Mean_{col}'] = round(mean_val, 6) if pd.notna(mean_val) else None
    
    results.append(row)

# 7. HASIL
df_result = pd.DataFrame(results)

# 8. SIMPAN
# df_result.to_excel('grouby_params.xlsx', index=False)
print("\n✅ File disimpan: grouby_params.xlsx")


✅ File disimpan: grouby_params.xlsx


# Join Table

### Joining Df_Results and Reel Data

In [ ]:
df_groupby = df_result.copy()
df_groupby.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,Mean_Flow Release,Mean_Jet Wire Ratio,Mean_Load KWH Refiner
0,2026-03-11 14:04:00,2026-03-11 15:05:00,61,17.986650,1159.935717,951.302131,7.979508,1310.224590,3.398197,30.404918,32.257213,0.94,216.331967
1,2026-03-11 15:05:00,2026-03-11 17:05:00,120,17.997198,1159.956710,951.197000,7.898667,1305.291500,3.400667,30.405417,32.257917,0.94,213.789333
2,2026-03-11 17:05:00,2026-03-11 18:17:00,72,17.997594,1159.948868,951.185972,7.899861,1305.660694,3.401528,30.404583,32.257778,0.94,211.020278
3,2026-03-11 18:17:00,2026-03-11 19:25:00,68,17.997967,1159.968028,951.197353,7.897941,1305.227647,3.398676,30.404853,32.257794,0.94,211.070147
4,2026-03-11 19:25:00,2026-03-11 20:35:00,70,17.903069,1159.937404,952.273000,7.899571,1306.115286,3.401000,30.405286,32.258143,0.94,212.930714


In [ ]:
df_reel = reel_pm17.copy()
df_reel.head()

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-01-01,07:15:00,1,2026-01-01 07:15:00,FC 12.2.Vp,30,12.94,0.97,492,193,147,0.298780,30,86.5,Acc Sotiss
1,2026-01-01,08:05:00,1,2026-01-01 08:05:00,FC 12.2.Vp,31,12.78,1.03,565,174,158,0.279646,32,86.9,Acc Sotiss
2,2026-01-01,08:55:00,1,2026-01-01 08:55:00,FC 12.2.Vp,32,12.92,0.97,538,203,128,0.237918,30,87,Acc Sotiss
3,2026-01-01,09:35:00,1,2026-01-01 09:35:00,FC 12.2.Vp,33,12.64,0.95,470,161,137,0.291489,30,87,Acc Sotiss
4,2026-01-01,10:15:00,1,2026-01-01 10:15:00,FC 12.2.Vp,34,12.28,0.96,378,131,105,0.277778,30,87.1,Acc Sotiss


In [ ]:
# Create Join Key in df_groupby
df_groupby['Join_Key_Timestamp'] = df_groupby['End_Time'] #End_Time
# Create Join Key in df_reel
df_reel['Join_Key_Timestamp'] = df_reel['Timestamp']

In [ ]:
# Join df_groupby with df_reel on Join_Key_Timestamp
df_joined = pd.merge(df_groupby, df_reel, left_on='Join_Key_Timestamp', right_on='Join_Key_Timestamp', how='inner')
df_joined.head()

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,...,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-03-11 14:04:00,2026-03-11 15:05:00,61,17.986650,1159.935717,951.302131,7.979508,1310.224590,3.398197,30.404918,...,96,15.96,0.96,1051,462,116,0.110371,28,79.9,Acc Sotiss
1,2026-03-11 15:05:00,2026-03-11 17:05:00,120,17.997198,1159.956710,951.197000,7.898667,1305.291500,3.400667,30.405417,...,97,15.80,0.99,1158,496,142,0.122625,27,79.8,Acc Sotiss
2,2026-03-11 17:05:00,2026-03-11 18:17:00,72,17.997594,1159.948868,951.185972,7.899861,1305.660694,3.401528,30.404583,...,98,15.88,0.98,1083,477,121,0.111727,28,79.8,Acc Sotiss
3,2026-03-11 18:17:00,2026-03-11 19:25:00,68,17.997967,1159.968028,951.197353,7.897941,1305.227647,3.398676,30.404853,...,99,15.84,0.99,1073,428,107,0.099720,26,80.7,Acc Sotiss
4,2026-03-11 19:25:00,2026-03-11 20:35:00,70,17.903069,1159.937404,952.273000,7.899571,1306.115286,3.401000,30.405286,...,1,15.78,0.98,1042,425,104,0.099808,24,80.4,Acc Sotiss


### Joining df_joined with BB Table

In [ ]:
df_joined.iloc[0:5,14:]

,Date,Time,Shift,Timestamp,Grade,Reel,Bw,Thickness,MDT,CDT,MDWT,RATIO MDWT (MDWT/MDT),MDS,Brightness,Insp. Status
0,2026-03-11,15:05:00,2,2026-03-11 15:05:00,T 15.2.Recycle,96,15.96,0.96,1051,462,116,0.110371,28,79.9,Acc Sotiss
1,2026-03-11,17:05:00,2,2026-03-11 17:05:00,T 15.2.Recycle,97,15.80,0.99,1158,496,142,0.122625,27,79.8,Acc Sotiss
2,2026-03-11,18:17:00,2,2026-03-11 18:17:00,T 15.2.Recycle,98,15.88,0.98,1083,477,121,0.111727,28,79.8,Acc Sotiss
3,2026-03-11,19:25:00,2,2026-03-11 19:25:00,T 15.2.Recycle,99,15.84,0.99,1073,428,107,0.099720,26,80.7,Acc Sotiss
4,2026-03-11,20:35:00,2,2026-03-11 20:35:00,T 15.2.Recycle,1,15.78,0.98,1042,425,104,0.099808,24,80.4,Acc Sotiss


In [ ]:
df_BB = pd.read_excel("../Efficiency/BB_Maret_PM17.xlsx", engine='openpyxl')
df_BB.drop(columns=['Grade'], inplace=True)
df_BB.head()

,Date,GSM,Total NBKP,Total LBKP,Total BB Recycle,Sub Total Pulp+Broke,% NBKP,% LBKP,% BB Recycle,% Pulp+Broke
0,2026-03-01,15,0,0,56277.840962,56277.840962,0,0,100,56277.840962
1,2026-03-02,15,0,0,54936.652290,54936.652290,0,0,100,111214.493252
2,2026-03-03,15,0,0,47692.563809,47692.563809,0,0,100,158907.057061
3,2026-03-04,"15/14,5",0,0,41482.083736,41482.083736,0,0,100,200389.140797
4,2026-03-05,"14,5",0,0,51059.006894,51059.006894,0,0,100,251448.147691


In [ ]:
# Standarisasi kolom "date" ke bentuk datetime
df_joined['Date'] = pd.to_datetime(df_joined['Date'])
df_BB['Date'] = pd.to_datetime(df_BB['Date'])

In [ ]:
# Joining
df_final = pd.merge(df_joined, df_BB, on='Date', how='inner') #inner
df_final.info()
df_final.head()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Start_Time              36 non-null     datetime64[us]
 1   End_Time                36 non-null     datetime64[us]
 2   Data_Count              36 non-null     int64         
 3   Mean_Creping            36 non-null     float64       
 4   Mean_Yankee Speed       36 non-null     float64       
 5   Mean_Pope Reel Speed    36 non-null     float64       
 6   Mean_Yankee Pressure    36 non-null     float64       
 7   Mean_Stock Flow         36 non-null     float64       
 8   Mean_Stock Consistency  36 non-null     float64       
 9   Mean_Flow Coating       36 non-null     float64       
 10  Mean_Flow Release       36 non-null     float64       
 11  Mean_Jet Wire Ratio     36 non-null     float64       
 12  Mean_Load KWH Refiner   36 non-null     float64       
 13  Joi

,Start_Time,End_Time,Data_Count,Mean_Creping,Mean_Yankee Speed,Mean_Pope Reel Speed,Mean_Yankee Pressure,Mean_Stock Flow,Mean_Stock Consistency,Mean_Flow Coating,...,Insp. Status,GSM,Total NBKP,Total LBKP,Total BB Recycle,Sub Total Pulp+Broke,% NBKP,% LBKP,% BB Recycle,% Pulp+Broke
0,2026-03-11 14:04:00,2026-03-11 15:05:00,61,17.986650,1159.935717,951.302131,7.979508,1310.224590,3.398197,30.404918,...,Acc Sotiss,"14,5/15",0,0,53227.077739,53227.077739,0,0,100,554710.450404
1,2026-03-11 15:05:00,2026-03-11 17:05:00,120,17.997198,1159.956710,951.197000,7.898667,1305.291500,3.400667,30.405417,...,Acc Sotiss,"14,5/15",0,0,53227.077739,53227.077739,0,0,100,554710.450404
2,2026-03-11 17:05:00,2026-03-11 18:17:00,72,17.997594,1159.948868,951.185972,7.899861,1305.660694,3.401528,30.404583,...,Acc Sotiss,"14,5/15",0,0,53227.077739,53227.077739,0,0,100,554710.450404
3,2026-03-11 18:17:00,2026-03-11 19:25:00,68,17.997967,1159.968028,951.197353,7.897941,1305.227647,3.398676,30.404853,...,Acc Sotiss,"14,5/15",0,0,53227.077739,53227.077739,0,0,100,554710.450404
4,2026-03-11 19:25:00,2026-03-11 20:35:00,70,17.903069,1159.937404,952.273000,7.899571,1306.115286,3.401000,30.405286,...,Acc Sotiss,"14,5/15",0,0,53227.077739,53227.077739,0,0,100,554710.450404
